# CPU testleri: eğitimden önce kodun doğrulanması

Bu notebook, `proje` dalındaki yeni kodu **GPU harcamadan** sınar. Colab'da **Çalışma zamanı → Çalışma zamanı türünü değiştir → CPU** seçin ve hücreleri sırayla çalıştırın (toplam yaklaşık 10–20 dakika).

| Bölüm | Ne test ediliyor? |
|---|---|
| (a) | DSRA birleştirmesinin ikili modda etkisiz olduğu bulgusu + kontrol koşulu |
| (b) | Hızlı (histogram tabanlı) mDice/mIoU'nun `eval.py` ile birebir aynı olduğu |
| (c) | `train_seed.py`'nin beş yapılandırmada da hatasız çalıştığı + ablasyonda arka plan başlarının gerçekten güncellenmediği |
| (d) | Aynı seed ile iki çalıştırmanın aynı kayıpları verdiği, farklı seed'in farklı kayıplar verdiği |

Her bölüm en sonda **TAMAM** ya da açık bir hata mesajı yazdırır.

## 0. Kurulum

Depoyu (`proje` dalı) klonlar, eksik paketleri yükler ve sürümleri yazdırır. **İlk satırdaki `GITHUB_USER` değerine kendi GitHub kullanıcı adınızı yazın.** Depo zaten klonlanmışsa en son değişiklikleri çeker (`git pull`).

In [ ]:
GITHUB_USER = 'KULLANICI_ADINIZ'   # <- kendi GitHub kullanıcı adınızı yazın
BRANCH = 'proje'

import os, sys, json, shutil, subprocess, platform
REPO_DIR = '/content/PraNet-V2'
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '-b', BRANCH,
                    f'https://github.com/{GITHUB_USER}/PraNet-V2.git', REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
os.chdir(f'{REPO_DIR}/binary_seg')
sys.path.insert(0, os.getcwd())
print(subprocess.run(['git', 'log', '--oneline', '-10'], capture_output=True, text=True).stdout)

In [ ]:
!pip -q install timm thop tabulate gdown
import torch, timm, scipy, numpy
print('Python', platform.python_version(), '| torch', torch.__version__, '| timm', timm.__version__,
      '| scipy', scipy.__version__, '| numpy', numpy.__version__)
print('CUDA var mı?', torch.cuda.is_available(), '(bu notebook CPU için; False olması normal)')

def run(args, tail=25):
    '''Bir betiği alt süreçte çalıştırır, çıktının son satırlarını gösterir, hata varsa durur.'''
    p = subprocess.run([sys.executable, '-W', 'ignore'] + args, capture_output=True, text=True)
    lines = (p.stdout + p.stderr).strip().splitlines()
    print('\n'.join(lines[-tail:]))
    if p.returncode != 0:
        raise RuntimeError('Komut hata verdi (çıkış kodu %d): %s' % (p.returncode, ' '.join(args)))
    return p

### Omurga ağırlıkları

Modeller kurulurken ImageNet ile ön-eğitilmiş Res2Net-50 ve PVTv2-B2 ağırlıklarını `binary_seg/models/` klasöründen yükler. Bu hücre onları resmî README'deki Google Drive bağlantılarından indirir (yaklaşık 100 MB). Drive "kota aşıldı" hatası verirse dosyaları tarayıcıdan indirip bu klasöre elle yükleyin.

In [ ]:
import gdown
os.makedirs('models', exist_ok=True)
BACKBONES = {
    'models/pvt_v2_b2.pth': '1FOv-3T91gJoB153XDUTAg4E0CGPHVQS4',
    'models/res2net50_v1b_26w_4s-3cf99910.pth': '1DzQfXikcSObsL98RmFNHR_7Wuy1vED8z',
}
for path, file_id in BACKBONES.items():
    if not os.path.exists(path):
        gdown.download(id=file_id, output=path, quiet=False)
    print(f'{path}: {os.path.getsize(path) / 1e6:.1f} MB')

## (a) DSRA birleştirmesi ikili modda etkisiz mi?

**Ne test ediliyor?** `lib/pranet.py`'de her DSRA aşaması, önceki aşamanın tahminini `softmax(crop_fg - crop_bg, dim=1)` ile ekliyor. Polip modelleri tek kanallı (`num_class=1`) olduğu için bu softmax her zaman 1'e eşit olmalı. Yani önceki aşamanın tahmini sonraki çıktıları **hiç etkilememeli**.

**Nasıl?** Kaba çözücü çıktısına (map5'i üreten `agg1`) büyük bir gürültü eklenir ve map2/map3/map4'ün ne kadar değiştiği ölçülür.
- `use_softmax=True` (resmî ayar): map2–map4 değişimi ≈ 0 olmalı, map5 ise değişmeli.
- `use_softmax=False` (kontrol): map2–map4 **değişmeli**. Bu, eklenen gürültünün sonraki aşamalara gerçekten ulaşabildiğini gösterir; yani ilk koşuldaki sıfır değişimin sebebi testin kendisi değildir.

Her iki omurga için ayrı ayrı çalıştırılır. Beklenen son satır: **CONFIRMED**.

In [ ]:
for mt in ['PraNet-V2', 'PVT-PraNet-V2']:
    print('=' * 20, mt, '=' * 20)
    run(['check_dsra_noop.py', '--model_type', mt], tail=16)
    print()

## (b) Hızlı mDice/mIoU, `eval.py` ile aynı sonucu veriyor mu?

**Ne test ediliyor?** `proje_common.image_metrics`, Dice ve IoU'yu 256 eşiğin her birinde piksel piksel döngüyle değil, gri seviye histogramlarıyla hesaplıyor (çok daha hızlı). Bu hücre, sonucun resmî `eval.py`'deki `eval_for_testAllInOne` fonksiyonuyla **birebir aynı** olduğunu gösterir. Diğer metrikler (Sm, wFm, meanEm, mae) zaten resmî fonksiyonlarla hesaplandığı için onlar da karşılaştırılır.

**Test durumları:** gerçekçi elips polipler, tamamen rastgele maskeler, boş tahmin (model hiçbir şey bulamıyor), boş maske (görüntüde polip yok) ve 0/255 dışı değerler içeren maskeler (`eval.py` > 0.5 olan her pikseli polip sayar; bu davranışın da korunduğu sınanır).

Not: Küme düzeyindeki mDice, görüntü başına değerlerin ortalamasıdır (`eval.py`'deki iki ortalama da doğrusal olduğu için sıra fark etmez). Bu yüzden görüntü düzeyinde eşitlik, küme düzeyinde eşitliği de garanti eder.

Beklenen: her metrik için en büyük fark < 1e-10 ve **TAMAM**.

In [ ]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from eval import eval_for_testAllInOne
from proje_common import image_metrics

METRICS = ['meanDic', 'meanIoU', 'Sm', 'wFm', 'meanEm', 'mae']
rng = np.random.default_rng(0)

def ellipse(h, w):
    yy, xx = np.mgrid[:h, :w]
    cy, cx = rng.uniform(0.3, 0.7) * h, rng.uniform(0.3, 0.7) * w
    ry, rx = rng.uniform(0.05, 0.3) * h, rng.uniform(0.05, 0.3) * w
    return ((yy - cy) / ry) ** 2 + ((xx - cx) / rx) ** 2 < 1

def make_case(kind, h=120, w=160):
    m = ellipse(h, w)
    if kind == 'gercekci':       # elips polip + gürültülü yumuşak tahmin
        gt = m.astype(np.uint8) * 255
        pred = np.clip(0.7 * m + rng.normal(0.1, 0.25, (h, w)), 0, 1)
    elif kind == 'rastgele':     # tamamen rastgele maske ve tahmin
        gt = (rng.random((h, w)) > 0.7).astype(np.uint8) * 255
        pred = rng.random((h, w))
    elif kind == 'bos_tahmin':   # model hiçbir şey bulamıyor
        gt = m.astype(np.uint8) * 255
        pred = np.zeros((h, w))
    elif kind == 'bos_maske':    # görüntüde polip yok
        gt = np.zeros((h, w), np.uint8)
        pred = rng.random((h, w))
    elif kind == 'gri_maske':    # maskede 1 ve 128 gibi ara değerler
        gt = (m * rng.choice([1, 128, 255], size=(h, w))).astype(np.uint8)
        pred = np.clip(0.6 * m + rng.normal(0.2, 0.3, (h, w)), 0, 1)
    return (pred * 255).astype(np.uint8), gt

kinds = ['gercekci'] * 6 + ['rastgele'] * 3 + ['bos_tahmin', 'bos_maske', 'gri_maske', 'gri_maske']
max_diff = {k: 0.0 for k in METRICS}
for kind in kinds:
    pred, gt = make_case(kind)
    ref = dict(zip(METRICS, eval_for_testAllInOne({'metrics': METRICS}, pred, gt)))
    ours = image_metrics(pred, gt, full=True)
    for k in METRICS:
        a, b = float(ref[k]), float(ours[k])
        if np.isnan(a) and np.isnan(b):
            continue                       # ikisi de tanımsız (örn. boş maskede wFm): eşit sayılır
        d = np.inf if (np.isnan(a) or np.isnan(b)) else abs(a - b)
        max_diff[k] = max(max_diff[k], d)
    print(f'{kind:<11} eval.py mDice={ref["meanDic"]:.6f} | bizim mDice={ours["meanDic"]:.6f}')

print()
for k, v in max_diff.items():
    print(f'{k:<8} en büyük fark: {v:.2e}')
assert all(v < 1e-10 for v in max_diff.values()), 'FARK VAR: hesap eval.py ile uyuşmuyor!'
print('\nTAMAM: Tüm metrikler eval.py ile aynı (fark < 1e-10).')

## (c) `train_seed.py` beş yapılandırmada çalışıyor mu?

**Ne test ediliyor?** Eğitim betiğinin uçtan uca hatasız çalıştığı: veri yükleme, model kurma, çok ölçekli ileri/geri geçiş, kayıp, ağırlık kaydı (`last.pth`) ve JSON kaydı. Beş yapılandırma: PraNet-V1, PVT-PraNet-V1, PraNet-V2, PVT-PraNet-V2 ve ablasyon (PVT-PraNet-V2, `--bg_weight 0`).

**Nasıl?** Önce 8 görüntülük sentetik bir eğitim kümesi üretilir (renkli elips "polipler"). Her yapılandırma 1 epoch'ta yalnızca 2 adım (`--smoke 2`) koşulur. CPU belleği için batch boyutu 2'dir; betiğin "effective batch size" uyarısı bu yüzden **beklenen** bir uyarıdır. Bu sonuçların bilimsel bir anlamı yoktur, yalnızca kodun çalıştığını gösterir.

Beklenen: her yapılandırma için sonlu kayıp değerleri ve en sonda **TAMAM**.

In [ ]:
from PIL import Image
TINY = '/content/tiny/TrainDataset'
shutil.rmtree('/content/tiny', ignore_errors=True)
os.makedirs(f'{TINY}/images'); os.makedirs(f'{TINY}/masks')
for i in range(8):
    h, w = 288, 384
    m = ellipse(h, w)
    img = (rng.random((h, w, 3)) * 80 + 100).astype(np.uint8)
    img[m] = img[m] * 0.5 + np.array([180, 60, 60]) * 0.5   # polipi farklı renkte yap
    Image.fromarray(img).save(f'{TINY}/images/{i:03d}.png')
    Image.fromarray((m * 255).astype(np.uint8)).save(f'{TINY}/masks/{i:03d}.png')
print(len(os.listdir(f'{TINY}/images')), 'sentetik görüntü hazır')

# Ortak argümanlar: 1 epoch, 2 adım, batch 2, alt süreç yok, ara kayıt yok
COMMON = ['--train_path', TINY, '--epochs', '1', '--smoke', '2', '--batchsize', '2',
          '--num_workers', '0', '--save_every', '0']
CONFIGS = [('PraNet-V1', 0.8, 'v1_res'), ('PVT-PraNet-V1', 0.8, 'v1_pvt'),
           ('PraNet-V2', 0.8, 'v2_res'), ('PVT-PraNet-V2', 0.8, 'v2_pvt'),
           ('PVT-PraNet-V2', 0.0, 'v2_pvt_ablasyon')]

for mt, bg, name in CONFIGS:
    print(f'===== {name}  ({mt}, bg_weight={bg}) =====')
    out = f'/content/smoke/{name}'
    shutil.rmtree(out, ignore_errors=True)
    run(['train_seed.py', '--model_type', mt, '--bg_weight', str(bg), '--seed', '1',
         '--out_dir', out] + COMMON, tail=4)
    log = json.load(open(f'{out}/train_log.json'))
    losses = [s[3] for s in log['first_steps']]
    assert os.path.exists(f'{out}/last.pth'), 'last.pth kaydedilmedi'
    assert len(losses) == 6 and all(np.isfinite(losses)), 'kayıplar eksik ya da sonlu değil'
    print(f'   6 adım kaybı (2 batch x 3 ölçek): {[round(l, 4) for l in losses]}')
    print(f'   parametre sayısı: {log["n_params"]:,} | cihaz: {log["gpu"]}\n')
print('TAMAM: Beş yapılandırmanın hepsi hatasız çalıştı.')

### (c-2) Ablasyonda arka plan başları gerçekten donuk mu?

**Ne test ediliyor?** Ablasyonun tasarlandığı gibi çalıştığı. `--bg_weight 0` ile arka plan kaybı hiç hesaplanmaz. (a)'daki bulgu doğruysa arka plan başlarının çıktıya başka bir yoldan da katkısı yoktur. Bu durumda bu başlar **hiç gradyan almamalı** ve ağırlıkları başlangıç değerinde kalmalıdır.

**Nasıl?** Aynı seed ile modelin başlangıç ağırlıkları yeniden üretilir ve adı `_bg` içeren tüm parametreler eğitim sonrası ağırlıklarla karşılaştırılır.
- Ablasyon: en büyük değişim **tam olarak 0** olmalı.
- Normal PVT-PraNet-V2 (kontrol): değişim **> 0** olmalı.

Not: Bu test, başlangıç ağırlıklarının aynı seed ile yeniden üretilebildiğini de dolaylı olarak doğrular.

In [ ]:
from proje_common import set_seed, build_model

def bg_change(out_dir, mt, seed=1):
    set_seed(seed, deterministic=True)
    init = build_model(mt)                       # eğitimin başlangıç ağırlıkları (aynı seed)
    trained = torch.load(f'{out_dir}/last.pth', map_location='cpu')
    diffs = [(trained[n] - p.detach()).abs().max().item()
             for n, p in init.named_parameters() if '_bg' in n]
    return max(diffs), len(diffs)

abl, n_abl = bg_change('/content/smoke/v2_pvt_ablasyon', 'PVT-PraNet-V2')
ref, n_ref = bg_change('/content/smoke/v2_pvt', 'PVT-PraNet-V2')
print(f'Ablasyon : {n_abl} arka plan parametresi, en büyük değişim = {abl:.3e}')
print(f'Normal V2: {n_ref} arka plan parametresi, en büyük değişim = {ref:.3e}')
assert abl == 0.0, 'Ablasyonda arka plan başları değişmiş: kayıp kapatılamamış olabilir'
assert ref > 0.0, 'Normal V2de arka plan başları değişmemiş: test hatalı olabilir'
print('\nTAMAM: Ablasyonda arka plan başları hiç güncellenmedi; normal V2de güncellendi.')

## (d) Aynı seed, aynı kayıplar mı?

**Ne test ediliyor?** Seed'in gerçekten tekrarlanabilirliği sağladığı. PraNet-V2 üç kez kısa koşulur:
- **A** ve **B**: aynı seed (7). İlk 6 adımın kayıpları **aynı** olmalı.
- **C**: farklı seed (8). Kayıplar **farklı** olmalı. Bu, seed'in model başlatmayı ve veri sırasını gerçekten değiştirdiğini gösterir.

CPU'da sonuçların bit düzeyinde aynı çıkması beklenir. GPU'da ise bilineer yeniden boyutlandırmanın geri yayılımı belirlenimci olmadığından çok küçük farklar (örneğin 1e-6 düzeyinde) çıkabilir. Bu beklenen bir durumdur ve önerideki sınırlılıklar bölümünde belirtilmiştir.

Beklenen: A–B farkı 0 (veya < 1e-6), A–C farkı belirgin ve **TAMAM**.

In [ ]:
runs = {'A': 7, 'B': 7, 'C': 8}
losses = {}
for name, seed in runs.items():
    out = f'/content/smoke/det_{name}'
    shutil.rmtree(out, ignore_errors=True)
    run(['train_seed.py', '--model_type', 'PraNet-V2', '--seed', str(seed), '--out_dir', out] + COMMON,
        tail=1)
    losses[name] = np.array([s[3] for s in json.load(open(f'{out}/train_log.json'))['first_steps']])

print(f"{'adım':<6}{'A (seed 7)':>14}{'B (seed 7)':>14}{'C (seed 8)':>14}")
for i in range(len(losses['A'])):
    print(f"{i + 1:<6}{losses['A'][i]:>14.6f}{losses['B'][i]:>14.6f}{losses['C'][i]:>14.6f}")
d_ab = np.abs(losses['A'] - losses['B']).max()
d_ac = np.abs(losses['A'] - losses['C']).max()
print(f'\nAynı seed (A-B) en büyük fark   : {d_ab:.3e}')
print(f'Farklı seed (A-C) en büyük fark : {d_ac:.3e}')
assert d_ab < 1e-6, 'Aynı seed farklı kayıp verdi: tekrarlanabilirlik sorunu'
assert d_ac > 1e-4, 'Farklı seed aynı kaybı verdi: seed etkisiz olabilir'
print('\nTAMAM: Aynı seed aynı kayıpları, farklı seed farklı kayıpları verdi.')

## Sonuç

Dört bölümün hepsi **TAMAM** yazdırdıysa kod GPU'da pilot eğitime hazırdır. Herhangi bir hücre hata verirse, hata mesajının tamamını kopyalayıp paylaşın; birlikte düzeltelim.

Bir sonraki adım (GPU pilotu): gerçek veri kümesinin indirilmesi, yayınlanmış ağırlıklarla ölçüm hattının doğrulanması ve tek bir tam eğitimin süresinin ölçülmesi.